### Import Dependencies

In [ ]:
import openai
import os
import json

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client


### Download all data from Qdrant

In [ ]:
quadrant_client = QdrantClient(url="http://localhost:6333")


In [ ]:
all_points = quadrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
    )

In [ ]:
all_points[0][0].payload

In [ ]:
all_context = [{"id": data.payload["parent_asin"], "text":data.payload["preprocessed_description"]} for data in all_points[0]]

In [ ]:
all_context

### Render a prompt to generate synthetic Eval reference dataset

In [ ]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "The reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "The id of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            }
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questIons about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multipple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json. dumps (output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be adble to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""



In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
print(USER_PROMPT)

In [ ]:
response = openai.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
        ],
        reasoning_effort="none"
)
print(response.choices[0].message.content)

In [ ]:
response
# completion_tokens=3180, prompt_tokens=19401, total_tokens=22581

In [ ]:
json_output = response.choices[0].message.content

In [ ]:
json_output

In [ ]:
json_output = json.loads(response.choices[0].message.content)

In [ ]:
json_output

In [ ]:
len(json_output)

In [ ]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0
for item in json_output:
    if len (item ["chunk_ids"]) == 1: 
        single_chunk_counter += 1 
    elif len(item ["chunk_ids"]) > 1: 
        multiple_chunk_counter += 1
    else:
        impossible_counter += 1

### 

In [ ]:
print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Impossible questions: {impossible_counter}")

In [ ]:
points = quadrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(
                    value="B0C5XLH1HM"
                )
            )
        ]
    ),
)[0]

In [ ]:
points[0].payload

In [ ]:
def get_description(parent_asin: str) -> str:

    points = quadrant_client.scroll(
        collection_name="Amazon-items-collection-01",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(
                        value=parent_asin
                    )
                )
            ]
        ),
    )[0]

    if not points:
        raise ValueError(f"No product found for parent_asin={parent_asin!r}")
    return points[0].payload["preprocessed_description"]

In [ ]:
get_description("B09C965RV9")

In [ ]:
all_ids = {cid for item in json_output for cid in item["chunk_ids"]}

missing = []
for cid in all_ids:
    pts = quadrant_client.scroll(
        collection_name="Amazon-items-collection-01",
        scroll_filter=Filter(must=[FieldCondition(
            key="parent_asin", match=MatchValue(value=cid))]),
    )[0]
    if not pts:
        missing.append(cid)

print("Unresolvable chunk_ids:", missing)

### Create Eval dataset in LangSmith

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

import os



In [ ]:
ls_client = Client()

In [ ]:
dataset_name = "RAG-evaluation-dataset"
eval_dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="Eval dataset for RAG system"
)


In [ ]:
for item in json_output:
    if any(cid in missing for cid in item["chunk_ids"]):
        print(f"Skipping example with unresolvable ids: {item['question']!r}")
        continue
    ls_client.create_example(
        dataset_id=eval_dataset.id,
        inputs = {
            "question": item["question"],
        },
        outputs = {
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )

